# 03 — Exportación de datos para Power BI
## easyMoney | TFM Data Science & AI — Nuclio School

**Prerequisitos:** `01-eda.ipynb` y `02-eda-deep-dive.ipynb` ejecutados

**Objetivo:** Exportar los datos limpios (sin anomalías) en formato CSV
para construir el dashboard de BI en Power BI Desktop.

**Input:** `master_df_flags.parquet` — tabla maestra con flags de calidad

**Outputs (15 CSVs):**

| # | Archivo | Descripción |
|---|---------|-------------|
| 1 | `period_summary.csv` | Evolución temporal de KPIs (17 períodos) |
| 2 | `product_penetration.csv` | Penetración por producto y período |
| 3 | `client_profile.csv` | Perfil demográfico Mayo 2019 (442K clientes) |
| 4 | `kpi_summary.csv` | KPIs calculados por período |
| 5 | `product_penetration_long.csv` | Penetración por producto (formato largo) |
| 6 | `age_sort.csv` | Orden numérico para age_group en Power BI |
| 7 | `salary_sort.csv` | Orden numérico para salary_group en Power BI |
| 8 | `contracts_by_type.csv` | Nuevas contrataciones por tipo cliente y período |
| 9 | `segment_by_period.csv` | Evolución de segmentos por período |
| 10 | `revenue_by_period.csv` | Revenue estimado por período (17 meses) |
| 11 | `revenue_by_product.csv` | Revenue estimado por producto, Mayo 2019 |
| 12 | `revenue_by_region.csv` | Revenue estimado por región, Mayo 2019 |
| 13 | `product_family_map.csv` | Lookup: 14 productos → 5 familias |
| 14 | `revenue_by_family.csv` | Revenue agrupado por familia, Mayo 2019 |
| 15 | `penetration_by_family.csv` | Penetración agrupada por familia, Mayo 2019 |
| 16 | `region_map.csv` | Nombre de provincia |
---

In [1]:
# ── Setup: imports, paths y diccionarios de referencia ──────────────
import pandas as pd
import os

BASE = 'C:\\Users\\farno\\OneDrive\\Desktop\\Data science & AI - Nuclio School\\proyecto final TFM\\tfm-fintech-easymoney\\data\\processed\\'

DATA_PATH   = BASE + 'master_df_flags.parquet'
OUTPUT_PATH = BASE + 'powerbi\\'

df = pd.read_parquet(DATA_PATH)
os.makedirs(OUTPUT_PATH, exist_ok=True)

product_cols = ['short_term_deposit','loans','mortgage','funds','securities',
                'long_term_deposit','credit_card','payroll','pension_plan',
                'payroll_account','emc_account','debit_card','em_account_p',
                'em_acount']

last_partition = df['pk_partition'].max()

# ── Diccionario de etiquetas (usado en varias tablas) ──────────────────────
label_map = {
    'em_acount':          'Cuenta easyMoney',
    'payroll':            'Domiciliaciones',
    'em_account_p':       'Cuenta easyMoney+',
    'debit_card':         'Tarjeta débito',
    'credit_card':        'Tarjeta crédito',
    'payroll_account':    'Cuenta nómina',
    'emc_account':        'Cuenta Crypto',
    'short_term_deposit': 'Depósito C/P',
    'long_term_deposit':  'Depósito L/P',
    'pension_plan':       'Plan pensiones',
    'funds':              'Fondos inversión',
    'securities':         'Valores',
    'mortgage':           'Hipoteca',
    'loans':              'Préstamos'
}

# ── Precios unitarios estimados (usado en revenue) ─────────────────────────
price_map = {
    'em_acount':          10,
    'emc_account':        10,
    'payroll_account':    10,
    'payroll':            10,
    'debit_card':         10,
    'em_account_p':       10,
    'short_term_deposit': 40,
    'long_term_deposit':  40,
    'funds':              40,
    'securities':         40,
    'pension_plan':       40,
    'credit_card':        40,
    'loans':              60,
    'mortgage':           60
}

# ── Verificación ───────────────────────────────────────────────────────────
print(f"✓ Parquet cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"✓ Flags: {[c for c in df.columns if 'anomaly' in c]}")
print(f"✓ Última partición: {last_partition}")
print(f"✓ label_map: {len(label_map)} productos")
print(f"✓ price_map: {len(price_map)} productos")

✓ Parquet cargado: 5,962,924 filas × 37 columnas
✓ Flags: ['age_anomaly', 'deceased_anomaly', 'entry_date_anomaly']
✓ Última partición: 2019-05-28 00:00:00
✓ label_map: 14 productos
✓ price_map: 14 productos


In [2]:
# ── Exportar CSVs para Power BI ───────────────────────────────────────────
# Nota: period_summary y product_penetration usan df completo (incluye anomalías)
# Esto es intencional — muestran la evolución de TODA la base de clientes
# Para análisis de perfil y revenue, se usa df_clean (sin anomalías)
# Tabla 1: Resumen por período
period_summary_export = df.groupby('pk_partition').agg(
    total_clients        = ('pk_cid', 'count'),
    active_clients       = ('active_customer', 'sum'),
    new_clients          = ('is_new_client', 'sum'),
    new_contracts        = ('new_contracts', 'sum'),
    avg_products         = ('total_products', 'mean'),
    clients_0_products   = ('total_products', lambda x: (x==0).sum()),
    clients_1_product    = ('total_products', lambda x: (x==1).sum()),
    clients_2plus        = ('total_products', lambda x: (x>=2).sum()),
).reset_index()
period_summary_export['pk_partition'] = period_summary_export['pk_partition'].astype(str).str[:10]
period_summary_export.to_csv(OUTPUT_PATH + 'period_summary.csv', index=False)
print(f"✓ period_summary.csv — {period_summary_export.shape}")

# Tabla 2: Penetración por producto por período
product_penetration = df.groupby('pk_partition')[product_cols].mean().mul(100).round(2).reset_index()
product_penetration['pk_partition'] = product_penetration['pk_partition'].astype(str).str[:10]
product_penetration.to_csv(OUTPUT_PATH + 'product_penetration.csv', index=False)
print(f"✓ product_penetration.csv — {product_penetration.shape}")

# Tabla 3: Perfil cliente última partición
df_clean = df[~df[['age_anomaly','deceased_anomaly','entry_date_anomaly']].any(axis=1)]
client_profile = df_clean[df_clean['pk_partition'] == last_partition][[
    'pk_cid', 'segment', 'age_group', 'salary_group',
    'gender', 'region_code', 'country_id',
    'total_products', 'is_new_client',
    'client_age_months', 'active_customer'
] + product_cols].copy()
client_profile['pk_partition'] = str(last_partition)[:10]
client_profile.to_csv(OUTPUT_PATH + 'client_profile.csv', index=False)
print(f"✓ client_profile.csv — {client_profile.shape}")

# Tabla 4: KPIs resumen
kpi_summary = period_summary_export.copy()
kpi_summary['pct_new_clients'] = (kpi_summary['new_clients'] / kpi_summary['total_clients'] * 100).round(2)
kpi_summary['pct_0_products']  = (kpi_summary['clients_0_products'] / kpi_summary['total_clients'] * 100).round(2)
kpi_summary['pct_1_product']   = (kpi_summary['clients_1_product'] / kpi_summary['total_clients'] * 100).round(2)
kpi_summary['pct_crosssell']   = (kpi_summary['clients_2plus'] / kpi_summary['total_clients'] * 100).round(2)
kpi_summary.to_csv(OUTPUT_PATH + 'kpi_summary.csv', index=False)
print(f"✓ kpi_summary.csv — {kpi_summary.shape}")

print(f"\n✓ Todos los archivos exportados en: {OUTPUT_PATH}")
print(f"\nArchivos Power BI:")
for f in os.listdir(OUTPUT_PATH):
    size = os.path.getsize(OUTPUT_PATH + f) / 1024
    print(f"  {f:<35} {size:.1f} KB")

✓ period_summary.csv — (17, 9)
✓ product_penetration.csv — (17, 15)
✓ client_profile.csv — (442157, 26)
✓ kpi_summary.csv — (17, 13)

✓ Todos los archivos exportados en: C:\Users\farno\OneDrive\Desktop\Data science & AI - Nuclio School\proyecto final TFM\tfm-fintech-easymoney\data\processed\powerbi\

Archivos Power BI:
  age_sort.csv                        0.1 KB
  client_profile.csv                  42458.3 KB
  contracts_by_type.csv               1.1 KB
  kpi_summary.csv                     1.8 KB
  penetration_by_family.csv           0.2 KB
  period_summary.csv                  1.4 KB
  product_family_map.csv              0.5 KB
  product_penetration.csv             1.5 KB
  product_penetration_long.csv        0.5 KB
  revenue_by_family.csv               0.2 KB
  revenue_by_period.csv               0.6 KB
  revenue_by_product.csv              0.6 KB
  revenue_by_region.csv               0.8 KB
  salary_sort.csv                     0.1 KB
  segment_by_period.csv               1.7 KB


In [3]:
# ── Tabla 5: Penetración por producto (formato largo para Power BI) ────────
# Nota: label_map definido en cell 1 (Setup)

# Nota: df_clean definido en cell 2
anomaly_flags = ['age_anomaly', 'deceased_anomaly', 'entry_date_anomaly']
df_clean = df[~df[anomaly_flags].any(axis=1)]
df_last = df_clean[df_clean['pk_partition'] == last_partition]

penetration_long = pd.DataFrame({
    'producto': list(label_map.values()),
    'nombre_tecnico': list(label_map.keys()),
    'penetracion_pct': [df_last[col].mean() * 100 for col in label_map.keys()]
}).round(2).sort_values('penetracion_pct', ascending=False)

penetration_long.to_csv(OUTPUT_PATH + 'product_penetration_long.csv', index=False)

print(f"✓ product_penetration_long.csv — {penetration_long.shape}")
print(f"\nVista previa:")
print(penetration_long.to_string(index=False))

✓ product_penetration_long.csv — (14, 3)

Vista previa:
         producto     nombre_tecnico  penetracion_pct
 Cuenta easyMoney          em_acount            67.01
   Tarjeta débito         debit_card             9.78
    Cuenta nómina    payroll_account             6.00
    Cuenta Crypto        emc_account             5.59
   Plan pensiones       pension_plan             3.92
  Domiciliaciones            payroll             3.69
     Depósito L/P  long_term_deposit             1.38
  Tarjeta crédito        credit_card             1.09
          Valores         securities             0.40
 Fondos inversión              funds             0.30
        Préstamos              loans             0.01
         Hipoteca           mortgage             0.01
Cuenta easyMoney+       em_account_p             0.00
     Depósito C/P short_term_deposit             0.00


In [4]:
# ── Tablas adicionales para Power BI ──────────────────────────────────────

# Tabla 6: Sort orders para age_group y salary_group
age_sort = pd.DataFrame({
    'age_group': ['<18','18-24','25-35','35-45','45-55','55-65','65+'],
    'age_sort':  [1, 2, 3, 4, 5, 6, 7]
})
age_sort.to_csv(OUTPUT_PATH + 'age_sort.csv', index=False)
print(f"✓ age_sort.csv — {age_sort.shape}")

salary_sort = pd.DataFrame({
    'salary_group': ['sin_ingreso','<20k','20-40k','40-60k','60-80k','80-120k','120k+'],
    'salary_sort':  [1, 2, 3, 4, 5, 6, 7]
})
salary_sort.to_csv(OUTPUT_PATH + 'salary_sort.csv', index=False)
print(f"✓ salary_sort.csv — {salary_sort.shape}")

# Tabla 7: Nuevos vs existentes por período
contracts_type = df[df['pk_partition'] != df['pk_partition'].min()].groupby(
    ['pk_partition', 'is_new_client']
).agg(
    contracts=('new_contracts', 'sum'),
    client_count=('pk_cid', 'count')
).reset_index()
contracts_type['pk_partition'] = contracts_type['pk_partition'].astype(str).str[:10]
contracts_type['tipo'] = contracts_type['is_new_client'].map({0: 'Existente', 1: 'Nuevo'})
contracts_type.to_csv(OUTPUT_PATH + 'contracts_by_type.csv', index=False)
print(f"✓ contracts_by_type.csv — {contracts_type.shape}")

# Tabla 8: Segmentos por período
segment_period = df.groupby(['pk_partition', 'segment'])['pk_cid'].count().reset_index()
segment_period.columns = ['pk_partition', 'segment', 'clientes']
segment_period['pk_partition'] = segment_period['pk_partition'].astype(str).str[:10]
segment_period.to_csv(OUTPUT_PATH + 'segment_by_period.csv', index=False)
print(f"✓ segment_by_period.csv — {segment_period.shape}")

print(f"\n✓ Todos los archivos Power BI:")
for f in sorted(os.listdir(OUTPUT_PATH)):
    size = os.path.getsize(OUTPUT_PATH + f) / 1024
    print(f"  {f:<40} {size:.1f} KB")

✓ age_sort.csv — (7, 2)
✓ salary_sort.csv — (7, 2)
✓ contracts_by_type.csv — (32, 5)
✓ segment_by_period.csv — (51, 3)

✓ Todos los archivos Power BI:
  age_sort.csv                             0.1 KB
  client_profile.csv                       42458.3 KB
  contracts_by_type.csv                    1.1 KB
  kpi_summary.csv                          1.8 KB
  penetration_by_family.csv                0.2 KB
  period_summary.csv                       1.4 KB
  product_family_map.csv                   0.5 KB
  product_penetration.csv                  1.5 KB
  product_penetration_long.csv             0.5 KB
  revenue_by_family.csv                    0.2 KB
  revenue_by_period.csv                    0.6 KB
  revenue_by_product.csv                   0.6 KB
  revenue_by_region.csv                    0.8 KB
  salary_sort.csv                          0.1 KB
  segment_by_period.csv                    1.7 KB


## Criterio de precios para revenue estimado

En ausencia de datos de margen neto, se aplican precios fijos por producto.
Los mismos productos se agrupan en 5 familias para la página Productos del dashboard.

| Familia | Productos | Precio unitario |
|---|---|---|
| Cuenta | em_acount, emc_account, payroll_account, payroll, em_account_p | €10 |
| Tarjetas | debit_card (€10), credit_card (€40) | €10 / €40 |
| Plan pensiones | pension_plan | €40 |
| Inversión | short_term_deposit, long_term_deposit, funds, securities | €40 |
| Financiación | loans, mortgage | €60 |

> **Revenue total estimado Mayo 2019: €5,328,470** *(datos limpios, sin anomalías)*  
> **Revenue por cliente: €12.05**
> **Nota:** `label_map` y `price_map` definidos en Cell 1 (Setup).

In [5]:
# ── Tabla 9 & 10: Revenue estimado ────────────────────────────────────────
# Nota: price_map definido en cell 1 (Setup)

# ── Filtrar anomalías antes de calcular revenue ────────────────────────────
anomaly_flags = ['age_anomaly', 'deceased_anomaly', 'entry_date_anomaly']
df_clean = df[~df[anomaly_flags].any(axis=1)]

print(f"df total:  {len(df):,} registros")
print(f"df_clean:  {len(df_clean):,} registros")
print(f"Excluidos: {len(df) - len(df_clean):,} registros ({(len(df)-len(df_clean))/len(df)*100:.2f}%)")

# Revenue por período
revenue_period = []
for partition, group in df_clean.groupby('pk_partition'):
    total_rev = sum(group[col].sum() * price for col, price in price_map.items())
    revenue_period.append({
        'pk_partition': str(partition)[:10],
        'revenue_estimado': round(total_rev, 2),
        'total_clientes': len(group),
        'revenue_por_cliente': round(total_rev / len(group), 2)
    })

df_revenue = pd.DataFrame(revenue_period)
df_revenue.to_csv(OUTPUT_PATH + 'revenue_by_period.csv', index=False)
print(f"✓ revenue_by_period.csv — {df_revenue.shape}")
print(df_revenue.to_string(index=False))

# Revenue por producto última partición
df_last = df_clean[df_clean['pk_partition'] == last_partition]
revenue_product = pd.DataFrame({
    'producto': list(label_map.values()),
    'nombre_tecnico': list(label_map.keys()),
    'precio_unitario': [price_map.get(col, 10) for col in label_map.keys()],
    'clientes_con_producto': [int(df_last[col].sum()) for col in label_map.keys()],
}).assign(
    revenue_estimado=lambda x: x['clientes_con_producto'] * x['precio_unitario']
).sort_values('revenue_estimado', ascending=False)

revenue_product.to_csv(OUTPUT_PATH + 'revenue_by_product.csv', index=False)
print(f"\n✓ revenue_by_product.csv — {revenue_product.shape}")
print(revenue_product.to_string(index=False))

# Revenue por región última partición
revenue_region = df_last.copy()
revenue_region['revenue'] = sum(
    revenue_region[col] * price for col, price in price_map.items()
)
revenue_region = revenue_region.groupby('region_code').agg(
    clientes=('pk_cid', 'count'),
    revenue_estimado=('revenue', 'sum')
).reset_index().sort_values('revenue_estimado', ascending=False)
revenue_region['revenue_estimado'] = revenue_region['revenue_estimado'].round(2)
revenue_region.to_csv(OUTPUT_PATH + 'revenue_by_region.csv', index=False)
print(f"\n✓ revenue_by_region.csv — {revenue_region.shape}")

df total:  5,962,924 registros
df_clean:  5,946,652 registros
Excluidos: 16,272 registros (0.27%)
✓ revenue_by_period.csv — (17, 4)
pk_partition  revenue_estimado  total_clientes  revenue_por_cliente
  2018-01-28           3555090          239238                14.86
  2018-02-28           3651940          242286                15.07
  2018-03-28           3747500          244967                15.30
  2018-04-28           3830780          247162                15.50
  2018-05-28           3852280          249654                15.43
  2018-06-28           3962660          251787                15.74
  2018-07-28           4157600          336054                12.37
  2018-08-28           4269140          352129                12.12
  2018-09-28           4496600          372700                12.06
  2018-10-28           4750570          400059                11.87
  2018-11-28           4881700          415094                11.76
  2018-12-28           5016740          421395      

In [6]:
# ── Tabla 11: Familia Producto — categorización de productos ──────────────

familia_map = {
    'em_acount':          'Cuenta',
    'em_account_p':       'Cuenta',
    'emc_account':        'Cuenta',
    'payroll_account':    'Cuenta',
    'payroll':            'Cuenta',
    'debit_card':         'Tarjetas',
    'credit_card':        'Tarjetas',
    'pension_plan':       'Plan pensiones',
    'funds':              'Inversión',
    'securities':         'Inversión',
    'long_term_deposit':  'Inversión',
    'short_term_deposit': 'Inversión',
    'loans':              'Financiación',
    'mortgage':           'Financiación',
}

df_last = df_clean[df_clean['pk_partition'] == last_partition]

# ── CSV 11a: product_family_map.csv — tabla lookup (conectar en Power BI) ──
family_lookup = pd.DataFrame([
    {'nombre_tecnico': col,
     'familia':        fam,
     'producto':       label_map.get(col, col)}
    for col, fam in familia_map.items()
]).sort_values(['familia', 'producto'])

family_lookup.to_csv(OUTPUT_PATH + 'product_family_map.csv', index=False)
print(f'✓ product_family_map.csv — {family_lookup.shape}')
print(family_lookup.to_string(index=False))

# ── CSV 11b: revenue_by_family.csv — revenue por familia (mayo 2019) ───────
rows = []
for col, fam in familia_map.items():
    price    = price_map.get(col, 10)
    clientes = int(df_last[col].sum())
    rows.append({
        'familia':               fam,
        'nombre_tecnico':        col,
        'producto':              label_map.get(col, col),
        'precio_unitario':       price,
        'clientes_con_producto': clientes,
        'revenue_estimado':      clientes * price
    })

rev_detail = pd.DataFrame(rows)

rev_by_family = (
    rev_detail
    .groupby('familia')
    .agg(
        clientes_totales = ('clientes_con_producto', 'sum'),
        revenue_estimado = ('revenue_estimado',      'sum'),
        num_productos    = ('nombre_tecnico',         'count')
    )
    .reset_index()
    .sort_values('revenue_estimado', ascending=False)
)
rev_by_family['revenue_estimado'] = rev_by_family['revenue_estimado'].round(2)
rev_by_family.to_csv(OUTPUT_PATH + 'revenue_by_family.csv', index=False)

print(f'\n✓ revenue_by_family.csv — {rev_by_family.shape}')
print(rev_by_family.to_string(index=False))

# ── CSV 11c: penetration_by_family.csv ────────────────────────
df_last = df_clean[df_clean['pk_partition'] == last_partition]

pen_rows = []
for familia, cols_in_family in rev_detail.groupby('familia')['nombre_tecnico'].apply(list).items():
    mask = df_last[cols_in_family].any(axis=1)
    clientes_unicos = int(mask.sum())
    pen_rows.append({
        'familia': familia,
        'clientes_unicos': clientes_unicos,
        'penetracion_pct': round(clientes_unicos / len(df_last) * 100, 2)
    })

pen_by_family = (
    pd.DataFrame(pen_rows)
    .sort_values('penetracion_pct', ascending=False)
)
pen_by_family.to_csv(OUTPUT_PATH + 'penetration_by_family.csv', index=False)

print(f'✓ penetration_by_family.csv — {pen_by_family.shape}')
print(pen_by_family.to_string(index=False))

# ── Resumen final ───────────────────────────────────────────────────────────
print('\n' + '─'*60)
print('✓ 3 CSVs nuevos listos para Power BI:')
print('  product_family_map.csv    → lookup: conectar con otros CSVs')
print('  revenue_by_family.csv     → revenue agrupado por familia')
print('  penetration_by_family.csv → penetración agrupada por familia')
print('─'*60)
print('\nEsperado revenue_by_family:')
print('  Cuenta          ~3.640.050 €  (mayor volumen)')
print('  Plan pensiones    ~694.120 €  (mayor precio unitario)')
print('  Inversión         ~369.440 €')
print('  Tarjetas          ~624.650 €')
print('  Financiación        ~3.180 €')

✓ product_family_map.csv — (14, 3)
    nombre_tecnico        familia          producto
       emc_account         Cuenta     Cuenta Crypto
         em_acount         Cuenta  Cuenta easyMoney
      em_account_p         Cuenta Cuenta easyMoney+
   payroll_account         Cuenta     Cuenta nómina
           payroll         Cuenta   Domiciliaciones
          mortgage   Financiación          Hipoteca
             loans   Financiación         Préstamos
short_term_deposit      Inversión      Depósito C/P
 long_term_deposit      Inversión      Depósito L/P
             funds      Inversión  Fondos inversión
        securities      Inversión           Valores
      pension_plan Plan pensiones    Plan pensiones
       credit_card       Tarjetas   Tarjeta crédito
        debit_card       Tarjetas    Tarjeta débito

✓ revenue_by_family.csv — (5, 4)
       familia  clientes_totales  revenue_estimado  num_productos
        Cuenta            363894           3638940              5
Plan pensiones     

In [7]:
# ── Tabla 16: region_code → nombre de provincia ────────────────────────────

region_names = {
    '1': 'Álava', '2': 'Albacete', '3': 'Alicante', '4': 'Almería',
    '5': 'Ávila', '6': 'Badajoz', '7': 'Baleares', '8': 'Barcelona',
    '9': 'Burgos', '10': 'Cáceres', '11': 'Cádiz', '12': 'Castellón',
    '13': 'Ciudad Real', '14': 'Córdoba', '15': 'Coruña, A', '16': 'Cuenca',
    '17': 'Girona', '18': 'Granada', '19': 'Guadalajara', '20': 'Gipuzkoa',
    '21': 'Huelva', '22': 'Huesca', '23': 'Jaén', '24': 'León',
    '25': 'Lleida', '26': 'Rioja, La', '27': 'Lugo', '28': 'Madrid',
    '29': 'Málaga', '30': 'Murcia', '31': 'Navarra', '32': 'Ourense',
    '33': 'Asturias', '34': 'Palencia', '35': 'Palmas, Las', '36': 'Pontevedra',
    '37': 'Salamanca', '38': 'Tenerife', '39': 'Cantabria', '40': 'Segovia',
    '41': 'Sevilla', '42': 'Soria', '43': 'Tarragona', '44': 'Teruel',
    '45': 'Toledo', '46': 'Valencia', '47': 'Valladolid', '48': 'Bizkaia',
    '49': 'Zamora', '50': 'Zaragoza', '51': 'Ceuta', '52': 'Melilla',
    '-1': 'UNKNOWN'
}

region_map = pd.DataFrame([
    {'region_code': code, 'provincia': name}
    for code, name in region_names.items()
])

region_map.to_csv(OUTPUT_PATH + 'region_map.csv', index=False)
print(f'✓ region_map.csv — {region_map.shape}')
print(region_map.to_string(index=False))

✓ region_map.csv — (53, 2)
region_code   provincia
          1       Álava
          2    Albacete
          3    Alicante
          4     Almería
          5       Ávila
          6     Badajoz
          7    Baleares
          8   Barcelona
          9      Burgos
         10     Cáceres
         11       Cádiz
         12   Castellón
         13 Ciudad Real
         14     Córdoba
         15   Coruña, A
         16      Cuenca
         17      Girona
         18     Granada
         19 Guadalajara
         20    Gipuzkoa
         21      Huelva
         22      Huesca
         23        Jaén
         24        León
         25      Lleida
         26   Rioja, La
         27        Lugo
         28      Madrid
         29      Málaga
         30      Murcia
         31     Navarra
         32     Ourense
         33    Asturias
         34    Palencia
         35 Palmas, Las
         36  Pontevedra
         37   Salamanca
         38    Tenerife
         39   Cantabria
         40  